<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [1]:
from datetime import datetime
import geopandas as gpd
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

In this step, participants define project-specific variables such as user initials, study site names and date. These variables are later used to create standardized filenames and organize outputs consistently throughout the workflow. This helps establish good data management and reproducibility practices.
###### Enter your initials
>initials = " "

###### Enter your site name
>site = " "

In this line we are using the datetime.now to automatically adjust the datetime to the time we run this notebook.
###### Date
>date = datetime.now()

In this line, participants will generate a standardized version name for the postcard dataset using project-specific variables and the current date. The code combines the user initials, study site name, and formatted date into a single descriptive filename that can be used to organize outputs consistently throughout the workflow.

>version = f"{initials}-{site}-{date.strftime('%d%m%Y')}_postcard_4"

>print(version)

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard

<font color='blue'>1.1. Reading the datapoints using the "read_file" function

Your code goes in the cells below. Add more cells here by clicking on the + button above.

First let's try and read your geojson datapoints in using the geopandas (gpd) library and the read_file function  
>joined_df = gpd.read_file("training-data/initial-site-date_postcard_4-training.csv")

print the attributes in the dataset
>joined_df

This cell removes columns containing the x and y data (values) using the .drop(columns) function. 
>joined_df=joined_df.drop(columns=["y", "x"])

This cell displays the column names and the length of the extracted training dataset. This allows the user to inspect all available spectral bands, indices, and metadata variables that were extracted from the imagery.
>print(len(joined_df.columns))

>joined_df.columns

This cell counts the number of samples belonging to each class using value_counts(). The output helps evaluate whether the training dataset is balanced across land cover categories.
>print(joined_df['ClassId'].value_counts())

### Create Model and Train Random Forest Classifer
This cell trains a Random Forest machine learning model using the prepared training dataset.

This line imports the train_test_split function from the Scikit-learn machine learning library. The function is used to divide the dataset into separate training and testing datasets for model evaluation.
>from sklearn.model_selection import train_test_split

This line splits the dataset stored in joined_df into two parts. The variable training_data stores 80% of the dataset, which will be used to train the model, while test_data stores the remaining 20%, which will later be used to test model performance. The parameter random_state=1337 ensures that the dataset is split in the same way each time the notebook is run, making the results reproducible.
>training_data, test_data = train_test_split(joined_df, test_size=0.2, random_state=1337)

##### The classes are the first column
This line converts the training dataset into a NumPy array and extracts all values from the first column. The extracted values are stored in the variable classes, which contains the training labels used by the machine learning model.

>classes = np.array(training_data)[:, 0]

##### The observation data is everything after the second column
This line extracts all columns except the first column from the training dataset. The resulting variables are stored in observations and represent the feature data used to train the model.
>observations = np.array(training_data)[:, 1:]

This line creates a Random Forest classification model. The max_depth=None parameter allows the decision trees to grow without a predefined depth limit. The class_weight='balanced' parameter automatically adjusts the importance of each class based on the number of training samples, helping reduce class imbalance issues. The random_state=1337 parameter ensures that the model produces consistent results each time it is trained.
##### create model
>classifier = RandomForestClassifier(max_depth=None, class_weight='balanced', random_state=1337)

This line trains the Random Forest model using the observation variables stored in observations and the class labels stored in classes. During training, the algorithm learns patterns and relationships between the spectral data and land cover classes so that it can later classify unknown pixels in satellite imagery.
##### ...and fit it to the data
>model = classifier.fit(observations, classes)

This line creates the file path where the trained machine learning model will be saved.
##### Dynamically create the filename with f-string
>file_path = f"models/{version}-test.model"

This line saves the trained Random Forest model to disk using the joblib.dump() function.
##### Save the model
>joblib.dump(model, file_path)

### Run the prediction

This line extracts the actual class labels from the testing dataset. The test_data DataFrame is converted into a NumPy array, and all values from the first column are selected. These values represent the true land cover classes used to evaluate the accuracy of the machine learning model.
>test_actual = np.array(test_data)[:, 0]

This line generates predictions using the trained Random Forest model. The testing dataset is converted into a NumPy array, and all columns except the first column are selected because these columns contain the predictor variables. The model.predict() function then uses these variables to predict the land cover class for each sample in the testing dataset. The predicted results are stored in the variable test_predicted.
>test_predicted = model.predict(np.array(test_data)[:,1:])

This line identifies all unique class labels that appear in either the actual test labels or the predicted labels. The np.concatenate() function combines both arrays into a single array, while np.unique() extracts only the distinct class values. The sorted() function then arranges the class labels in order. This step ensures that all possible classes are included consistently during evaluation and comparison.
>unique_labels = sorted(np.unique(np.concatenate([np.asarray(test_actual), np.asarray(test_predicted)])))

This line creates a confusion matrix using the pd.crosstab() function. The table compares the actual class labels stored in test_actual against the predicted class labels stored in test_predicted. Each row represents the true class, while each column represents the predicted class. The parameter margins=True adds row and column totals to the table, making it easier to analyse overall classification performance and identify misclassified samples.
>pd.crosstab(test_actual, test_predicted, margins=True)

This line calculates the classification accuracy of the trained Random Forest model. The variable test_actual contains the true class labels from the testing dataset, while test_predicted contains the class labels predicted by the model. The accuracy_score() function compares both sets of values and computes the proportion of correctly classified samples. The resulting value ranges between 0 and 1, where values closer to 1 indicate higher classification accuracy and better model performance.
>accuracy_score(test_actual, test_predicted)